In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
Epsilon Sensitivity Analysis for KL and ML Decoders
====================================================
This script evaluates the impact of the smoothing parameter epsilon
on KL Divergence and Maximum Likelihood decoders for composite DNA.

Alphabet: A_6 (2mix_only, 10 classes)
Error Model: Erlich (EZ17)
Coverage Levels: M ∈ {1, 2, 3, 5, 8, 10, 15, 20, 25}
Epsilon Values: {1e-10, 1e-5, 1e-3, 0.01, 0.05, 0.1}
"""

# =============================================================================
# IMPORTS
# =============================================================================
import os
import random
import pickle
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time

import torch
from torch.utils.data import Dataset, DataLoader, random_split

# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================
DEVICE_ID = "0"  # Change this based on your GPU availability
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # Fixed settings for this analysis
    "error_model": "erlich",
    "error_name": "EZ17",
    "alphabet_mode": "2mix_only",
    "vocab_size": 10,
    "seq_length": 136,
    
    # Dataset parameters
    "num_samples": 100000,
    "max_coverage": 25,
    
    # Coverage levels to test
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25],
    
    # Epsilon values to test
    "epsilon_values": [1e-10, 1e-5, 1e-3, 0.01, 0.05, 0.1],
    
    # Evaluation parameters
    "batch_size": 500,
    
    # Reproducibility
    "seed": 42,
    
    # Output directory
    "results_dir": "./epsilon_sensitivity_results"
}

# Build dataset path
CONFIG["dataset_dir"] = "./dataset"
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"dna_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print(f"📋 EPSILON SENSITIVITY ANALYSIS CONFIGURATION")
print(f"{'='*60}")
print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")
print(f"   Error Model: {CONFIG['error_name']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")
print(f"   Epsilon Values: {CONFIG['epsilon_values']}")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"{'='*60}")

# =============================================================================
# SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")

# =============================================================================
# SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================
def build_symbol_to_idx():
    """Build symbol-to-index mapping for 2mix_only alphabet."""
    symbol_to_idx = {
        'A': 0, 'C': 1, 'G': 2, 'T': 3,
        'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9
    }
    return symbol_to_idx

def build_ideal_vectors():
    """Build ideal frequency vectors for 2mix_only alphabet."""
    ideal_vectors = [
        # Pure bases (indices 0-3)
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        
        # Two-mix (indices 4-9) - uniform 0.5/0.5
        [0.5, 0.0, 0.0, 0.5],  # M1 (A|T)
        [0.0, 0.5, 0.5, 0.0],  # M2 (C|G)
        [0.0, 0.5, 0.0, 0.5],  # M3 (C|T)
        [0.0, 0.0, 0.5, 0.5],  # M4 (G|T)
        [0.5, 0.5, 0.0, 0.0],  # M5 (A|C)
        [0.5, 0.0, 0.5, 0.0],  # M6 (A|G)
    ]
    return torch.tensor(ideal_vectors, dtype=torch.float32)

SYMBOL_TO_IDX = build_symbol_to_idx()
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors().to(device)

print(f"\n📊 Symbol Mappings (2mix_only):")
print(f"   {'Symbol':<8} {'Index':<6} {'Ideal Vector [A, C, G, T]'}")
print(f"   {'-'*50}")
for sym, idx in sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<8} {idx:<6} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]")

# =============================================================================
# DATA PREPROCESSING
# =============================================================================
def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix

# =============================================================================
# PYTORCH DATASET CLASS
# =============================================================================
class CompositeDNADataset(Dataset):
    """PyTorch Dataset for Composite DNA data."""
    
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)

# =============================================================================
# KL AND ML DECODERS WITH VARIABLE EPSILON
# =============================================================================
def kl_divergence_decoder(obs, ideal_vectors, epsilon):
    """
    KL Divergence Decoder with specified epsilon.
    
    Args:
        obs: (Batch, L, 4) observed frequency vectors
        ideal_vectors: (num_classes, 4) ideal frequency vectors
        epsilon: Smoothing parameter
    
    Returns:
        (Batch, L) predicted class indices
    """
    # Smooth ideal vectors
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    # Cross-entropy: -Σ P_obs * log(P_ideal)
    obs_expanded = obs.unsqueeze(2)  # (Batch, L, 1, 4)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)  # (1, 1, C, 4)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)  # (Batch, L, C)
    
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon):
    """
    Maximum Likelihood Decoder with specified epsilon.
    
    Args:
        obs: (Batch, L, 4) observed frequency vectors
        ideal_vectors: (num_classes, 4) ideal frequency vectors
        epsilon: Smoothing parameter
    
    Returns:
        (Batch, L) predicted class indices
    """
    # Smooth ideal vectors
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    # Log-likelihood: Σ P_obs * log(P_ideal)
    obs_expanded = obs.unsqueeze(2)  # (Batch, L, 1, 4)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)  # (1, 1, C, 4)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)  # (Batch, L, C)
    
    return torch.argmax(log_likelihood, dim=-1)

# =============================================================================
# EVALUATION FUNCTION
# =============================================================================
def evaluate_decoders_with_epsilon(loader, ideal_vectors, epsilon, device):
    """
    Evaluate KL and ML decoders with a specific epsilon value.
    
    Args:
        loader: DataLoader
        ideal_vectors: Ideal frequency vectors
        epsilon: Smoothing parameter
        device: torch device
    
    Returns:
        Dictionary with accuracy for KL and ML decoders
    """
    correct_kl = 0
    correct_ml = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (Batch, L, 4)
            
            # KL Divergence Decoder
            pred_kl = kl_divergence_decoder(obs, ideal_vectors, epsilon)
            
            # Maximum Likelihood Decoder
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors, epsilon)
            
            # Count correct
            total += labels.numel()
            correct_kl += (pred_kl == labels).sum().item()
            correct_ml += (pred_ml == labels).sum().item()
    
    accuracies = {
        'kl': 100 * correct_kl / total,
        'ml': 100 * correct_ml / total
    }
    return accuracies

# =============================================================================
# MAIN EXPERIMENT FUNCTION
# =============================================================================
def run_epsilon_sensitivity_analysis(config):
    """
    Run complete epsilon sensitivity analysis.
    """
    print(f"\n{'='*70}")
    print(f"🔬 EPSILON SENSITIVITY ANALYSIS")
    print(f"{'='*70}")
    
    # Check dataset exists
    if not os.path.exists(config['dataset_path']):
        raise FileNotFoundError(
            f"\n❌ Dataset not found: {config['dataset_path']}\n"
            f"   Please generate the dataset first."
        )
    
    # Load dataset info
    with open(config['dataset_path'], 'rb') as f:
        data = pickle.load(f)
    print(f"✅ Dataset loaded: {len(data['data']):,} samples")
    
    # Initialize results storage
    results = {
        'coverage': config['coverage_levels'],
        'epsilon_values': config['epsilon_values'],
        'kl': {},  # {epsilon: [acc_M1, acc_M2, ...]}
        'ml': {},  # {epsilon: [acc_M1, acc_M2, ...]}
    }
    
    # Initialize for each epsilon
    for eps in config['epsilon_values']:
        results['kl'][eps] = []
        results['ml'][eps] = []
    
    # Total experiments
    total_experiments = len(config['coverage_levels']) * len(config['epsilon_values'])
    experiment_count = 0
    
    start_time = time.time()
    
    # Loop over coverage levels
    for M in config['coverage_levels']:
        print(f"\n{'='*60}")
        print(f"📊 Coverage M = {M}")
        print(f"{'='*60}")
        
        # Create dataset with this coverage
        set_seed(config['seed'])
        full_ds = CompositeDNADataset(
            config['dataset_path'],
            config['seq_length'],
            SYMBOL_TO_IDX,
            limit_coverage=M
        )
        
        # Use only test set (20%)
        train_size = int(0.8 * len(full_ds))
        val_size = len(full_ds) - train_size
        _, val_ds = random_split(full_ds, [train_size, val_size])
        
        val_loader = DataLoader(val_ds, batch_size=config['batch_size'], 
                                shuffle=False, num_workers=0)
        
        print(f"   Test samples: {val_size:,}")
        
        # Evaluate for each epsilon
        for eps in config['epsilon_values']:
            experiment_count += 1
            
            accuracies = evaluate_decoders_with_epsilon(
                val_loader, IDEAL_VECTORS, eps, device
            )
            
            results['kl'][eps].append(accuracies['kl'])
            results['ml'][eps].append(accuracies['ml'])
            
            # Check equivalence
            diff = abs(accuracies['kl'] - accuracies['ml'])
            equiv_status = "✓ IDENTICAL" if diff < 1e-6 else f"⚠ DIFF={diff:.6f}"
            
            print(f"   ε={eps:<10} | KL: {accuracies['kl']:6.2f}% | ML: {accuracies['ml']:6.2f}% | {equiv_status}")
    
    elapsed = time.time() - start_time
    print(f"\n⏱️  Total time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    
    return results

# =============================================================================
# PRINT RESULTS TABLE
# =============================================================================
def print_results_table(results):
    """Print results in formatted tables."""
    
    print(f"\n{'='*80}")
    print(f"📊 COMPLETE RESULTS TABLE - KL Divergence Decoder")
    print(f"{'='*80}")
    
    # Header
    header = f"{'M':<6}"
    for eps in results['epsilon_values']:
        header += f" | ε={eps:<10}"
    print(header)
    print("-" * 80)
    
    # Data rows
    for i, M in enumerate(results['coverage']):
        row = f"{M:<6}"
        for eps in results['epsilon_values']:
            row += f" | {results['kl'][eps][i]:>12.2f}"
        print(row)
    
    print(f"\n{'='*80}")
    print(f"📊 COMPLETE RESULTS TABLE - Maximum Likelihood Decoder")
    print(f"{'='*80}")
    
    # Header
    print(header)
    print("-" * 80)
    
    # Data rows
    for i, M in enumerate(results['coverage']):
        row = f"{M:<6}"
        for eps in results['epsilon_values']:
            row += f" | {results['ml'][eps][i]:>12.2f}"
        print(row)
    
    # Verify equivalence
    print(f"\n{'='*80}")
    print(f"🔍 KL-ML EQUIVALENCE CHECK")
    print(f"{'='*80}")
    
    all_identical = True
    for eps in results['epsilon_values']:
        for i, M in enumerate(results['coverage']):
            diff = abs(results['kl'][eps][i] - results['ml'][eps][i])
            if diff > 1e-6:
                print(f"   ⚠ Difference at M={M}, ε={eps}: {diff:.8f}")
                all_identical = False
    
    if all_identical:
        print("   ✅ KL and ML decoders produce IDENTICAL results for all configurations!")
    
    # Best epsilon analysis
    print(f"\n{'='*80}")
    print(f"📈 BEST EPSILON ANALYSIS (Average Accuracy Across All M)")
    print(f"{'='*80}")
    
    for eps in results['epsilon_values']:
        avg_acc = np.mean(results['kl'][eps])
        print(f"   ε = {eps:<10} : Average Accuracy = {avg_acc:.2f}%")

# =============================================================================
# PLOTTING FUNCTION
# =============================================================================
def plot_epsilon_sensitivity(results, save_path, config):
    """
    Create epsilon sensitivity plots.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Color map for different epsilon values
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(results['epsilon_values'])))
    markers = ['o', 's', '^', 'D', 'v', 'p']
    
    # Plot 1: KL Decoder
    ax1 = axes[0]
    for i, eps in enumerate(results['epsilon_values']):
        label = f'ε = {eps}'
        ax1.plot(results['coverage'], results['kl'][eps], 
                 marker=markers[i % len(markers)], 
                 color=colors[i], 
                 linewidth=2, 
                 markersize=8,
                 label=label)
    
    ax1.set_xlabel('Coverage Depth (M)', fontsize=12)
    ax1.set_ylabel('Symbol Accuracy (%)', fontsize=12)
    ax1.set_title('KL Divergence Decoder: Epsilon Sensitivity', fontsize=14)
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 105)
    ax1.set_xticks(results['coverage'])
    
    # Plot 2: ML Decoder
    ax2 = axes[1]
    for i, eps in enumerate(results['epsilon_values']):
        label = f'ε = {eps}'
        ax2.plot(results['coverage'], results['ml'][eps], 
                 marker=markers[i % len(markers)], 
                 color=colors[i], 
                 linewidth=2, 
                 markersize=8,
                 label=label)
    
    ax2.set_xlabel('Coverage Depth (M)', fontsize=12)
    ax2.set_ylabel('Symbol Accuracy (%)', fontsize=12)
    ax2.set_title('Maximum Likelihood Decoder: Epsilon Sensitivity', fontsize=14)
    ax2.legend(fontsize=10, loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 105)
    ax2.set_xticks(results['coverage'])
    
    plt.suptitle(f"Epsilon Sensitivity Analysis - {config['alphabet_mode']} ({config['vocab_size']} classes), {config['error_name']} Error Model", 
                 fontsize=14, y=1.02)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n📈 Plot saved: {save_path}")


def plot_combined_epsilon_sensitivity(results, save_path, config):
    """
    Create a single combined plot showing epsilon sensitivity.
    """
    plt.figure(figsize=(12, 8))
    
    # Color map for different epsilon values
    colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']
    markers = ['o', 's', '^', 'D', 'v', 'p']
    linestyles = ['-', '--', '-.', ':', '-', '--']
    
    for i, eps in enumerate(results['epsilon_values']):
        # Use KL results (identical to ML)
        label = f'ε = {eps}'
        plt.plot(results['coverage'], results['kl'][eps], 
                 marker=markers[i], 
                 color=colors[i], 
                 linestyle=linestyles[i],
                 linewidth=2.5, 
                 markersize=10,
                 label=label)
    
    plt.xlabel('Coverage Depth (M)', fontsize=14)
    plt.ylabel('Symbol Accuracy (%)', fontsize=14)
    plt.title(f'Epsilon Sensitivity Analysis\n{config["alphabet_mode"]} ({config["vocab_size"]} classes), {config["error_name"]} Error Model', 
              fontsize=16)
    plt.legend(fontsize=12, loc='lower right', ncol=2)
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'], fontsize=12)
    plt.yticks(fontsize=12)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Combined plot saved: {save_path}")


def plot_epsilon_accuracy_at_fixed_M(results, save_path, config, fixed_M_values=[5, 10, 15, 25]):
    """
    Create bar plot showing accuracy vs epsilon at fixed coverage values.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    # Filter to available M values
    available_M = [M for M in fixed_M_values if M in results['coverage']]
    
    x = np.arange(len(results['epsilon_values']))
    width = 0.6
    
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(available_M)))
    
    for ax_idx, M in enumerate(available_M):
        ax = axes[ax_idx]
        M_idx = results['coverage'].index(M)
        
        accuracies = [results['kl'][eps][M_idx] for eps in results['epsilon_values']]
        
        bars = ax.bar(x, accuracies, width, color='steelblue', edgecolor='black')
        
        # Add value labels on bars
        for bar, acc in zip(bars, accuracies):
            height = bar.get_height()
            ax.annotate(f'{acc:.1f}%',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Epsilon (ε)', fontsize=12)
        ax.set_ylabel('Symbol Accuracy (%)', fontsize=12)
        ax.set_title(f'Coverage M = {M}', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels([str(eps) for eps in results['epsilon_values']], fontsize=10)
        ax.set_ylim(0, max(accuracies) * 1.15)
        ax.grid(True, alpha=0.3, axis='y')
    
    # Hide unused subplots
    for ax_idx in range(len(available_M), 4):
        axes[ax_idx].set_visible(False)
    
    plt.suptitle(f'Accuracy vs Epsilon at Different Coverage Depths\n{config["alphabet_mode"]} ({config["vocab_size"]} classes), {config["error_name"]}', 
                 fontsize=16)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Bar plot saved: {save_path}")

# =============================================================================
# SAVE RESULTS TO FILE
# =============================================================================
def save_results_to_file(results, save_path):
    """Save results to a text file."""
    with open(save_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("EPSILON SENSITIVITY ANALYSIS RESULTS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Alphabet: 2mix_only (10 classes)\n")
        f.write(f"Error Model: EZ17 (Erlich)\n")
        f.write(f"Coverage Levels: {results['coverage']}\n")
        f.write(f"Epsilon Values: {results['epsilon_values']}\n\n")
        
        # KL Results Table
        f.write("="*80 + "\n")
        f.write("KL DIVERGENCE DECODER ACCURACY (%)\n")
        f.write("="*80 + "\n")
        
        header = f"{'M':<6}"
        for eps in results['epsilon_values']:
            header += f" | ε={eps:<10}"
        f.write(header + "\n")
        f.write("-"*80 + "\n")
        
        for i, M in enumerate(results['coverage']):
            row = f"{M:<6}"
            for eps in results['epsilon_values']:
                row += f" | {results['kl'][eps][i]:>12.2f}"
            f.write(row + "\n")
        
        f.write("\n")
        
        # ML Results Table
        f.write("="*80 + "\n")
        f.write("MAXIMUM LIKELIHOOD DECODER ACCURACY (%)\n")
        f.write("="*80 + "\n")
        
        f.write(header + "\n")
        f.write("-"*80 + "\n")
        
        for i, M in enumerate(results['coverage']):
            row = f"{M:<6}"
            for eps in results['epsilon_values']:
                row += f" | {results['ml'][eps][i]:>12.2f}"
            f.write(row + "\n")
        
        f.write("\n")
        
        # Best epsilon summary
        f.write("="*80 + "\n")
        f.write("AVERAGE ACCURACY BY EPSILON\n")
        f.write("="*80 + "\n")
        
        for eps in results['epsilon_values']:
            avg_acc = np.mean(results['kl'][eps])
            f.write(f"ε = {eps:<12} : {avg_acc:.2f}%\n")
        
        f.write("\n")
        f.write("Note: KL and ML decoders produce identical results.\n")
    
    print(f"📄 Results saved to: {save_path}")

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    
    print("\n" + "="*70)
    print("🔬 EPSILON SENSITIVITY ANALYSIS FOR KL/ML DECODERS")
    print("="*70)
    print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")
    print(f"   Error Model: {CONFIG['error_name']}")
    print(f"   Coverage Levels: {CONFIG['coverage_levels']}")
    print(f"   Epsilon Values: {CONFIG['epsilon_values']}")
    print("="*70)
    
    # Run analysis
    results = run_epsilon_sensitivity_analysis(CONFIG)
    
    # Print results table
    print_results_table(results)
    
    # Save results to file
    results_file = os.path.join(CONFIG['results_dir'], "epsilon_sensitivity_results.txt")
    save_results_to_file(results, results_file)
    
    # Create plots
    plot_path_dual = os.path.join(CONFIG['results_dir'], "epsilon_sensitivity_dual_plot.png")
    plot_epsilon_sensitivity(results, plot_path_dual, CONFIG)
    
    plot_path_combined = os.path.join(CONFIG['results_dir'], "epsilon_sensitivity_combined_plot.png")
    plot_combined_epsilon_sensitivity(results, plot_path_combined, CONFIG)
    
    plot_path_bars = os.path.join(CONFIG['results_dir'], "epsilon_sensitivity_bar_plot.png")
    plot_epsilon_accuracy_at_fixed_M(results, plot_path_bars, CONFIG)
    
    print(f"\n{'='*70}")
    print(f"✅ ANALYSIS COMPLETE!")
    print(f"{'='*70}")
    print(f"   Results directory: {CONFIG['results_dir']}")
    print(f"   - epsilon_sensitivity_results.txt")
    print(f"   - epsilon_sensitivity_dual_plot.png")
    print(f"   - epsilon_sensitivity_combined_plot.png")
    print(f"   - epsilon_sensitivity_bar_plot.png")
    print(f"{'='*70}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080

📋 EPSILON SENSITIVITY ANALYSIS CONFIGURATION
   Alphabet: 2mix_only (10 classes)
   Error Model: EZ17
   Sequence Length: 136
   Coverage Levels: [1, 2, 3, 5, 8, 10, 15, 20, 25]
   Epsilon Values: [1e-10, 1e-05, 0.001, 0.01, 0.05, 0.1]
   Dataset Path: ./dataset/dna_EZ17_2mix_only_100000_25.pkl
🎲 Random seed set to: 42

📊 Symbol Mappings (2mix_only):
   Symbol   Index  Ideal Vector [A, C, G, T]
   --------------------------------------------------
   A        0      [1.00, 0.00, 0.00, 0.00]
   C        1      [0.00, 1.00, 0.00, 0.00]
   G        2      [0.00, 0.00, 1.00, 0.00]
   T        3      [0.00, 0.00, 0.00, 1.00]
   M1       4      [0.50, 0.00, 0.00, 0.50]
   M2       5      [0.00, 0.50, 0.50, 0.00]
   M3       6      [0.00, 0.50, 0.00, 0.50]
   M4       7      [0.00, 0.00, 0.50, 0.50]
   M5       8      [0.50, 0.50, 0.00, 0.00]
   M6       9      [0.50, 0.00, 0.50, 0.00]

🔬 EPSILON SENSITIVITY ANALYSIS FOR KL/ML DECODERS
   


📈 Plot saved: ./epsilon_sensitivity_results/epsilon_sensitivity_dual_plot.png
📈 Combined plot saved: ./epsilon_sensitivity_results/epsilon_sensitivity_combined_plot.png
📈 Bar plot saved: ./epsilon_sensitivity_results/epsilon_sensitivity_bar_plot.png

✅ ANALYSIS COMPLETE!
   Results directory: ./epsilon_sensitivity_results
   - epsilon_sensitivity_results.txt
   - epsilon_sensitivity_dual_plot.png
   - epsilon_sensitivity_combined_plot.png
   - epsilon_sensitivity_bar_plot.png
